# Merge 5 TypePro shards and publish the final dataset

Settings: **Internet ON**, accelerator **None/CPU**. Add secrets
`KAGGLE_USERNAME` and `KAGGLE_KEY`. Run only after all `5` private
shard datasets have been published successfully.


In [ ]:
SHARD_COUNT = 5
REPOSITORY = 'https://github.com/duyvu1105/TypePro.git'
BRANCH = 'main'
FINAL_DATASET_SLUG = "typepro-python-contrastive"
SEED = 13

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
os.environ["PYTHONUNBUFFERED"] = "1"

REPO_DIR = Path("/kaggle/working/TypePro")
DOWNLOAD_DIR = Path("/kaggle/working/downloaded_shards")
MERGED_BUILD = Path("/kaggle/working/typepro_build")
FINAL_DIR = Path("/kaggle/working/typepro_python_contrastive")

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(value) for value in command], cwd=cwd, check=True)


## Clone code and install dependencies


In [ ]:
if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, REPO_DIR])
PIPELINE_DIR = REPO_DIR / "codet5p_type_retrieval"
run([sys.executable, "-m", "pip", "install", "-q", "-r", PIPELINE_DIR / "requirements-build.txt"])


## Download and extract every private shard dataset


In [ ]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
shard_builds = []
for index in range(SHARD_COUNT):
    dataset_id = f"{os.environ['KAGGLE_USERNAME']}/typepro-build-shard-{index:02d}"
    target = DOWNLOAD_DIR / f"shard_{index:02d}"
    target.mkdir(parents=True, exist_ok=True)
    run(["kaggle", "datasets", "download", "-d", dataset_id, "-p", target, "--unzip"])
    archives = list(target.glob("typepro_build_shard_*.zip"))
    if len(archives) != 1:
        raise RuntimeError(f"{dataset_id}: expected one build archive, found {archives}")
    with zipfile.ZipFile(archives[0]) as bundle:
        bundle.extractall(target)
    builds = list(target.glob("typepro_build_shard_*"))
    if len(builds) != 1:
        raise RuntimeError(f"{dataset_id}: cannot locate extracted build directory")
    marker = json.loads((builds[0] / "shard_manifest.json").read_text(encoding="utf-8"))
    if marker["shard_index"] != index or marker["shard_count"] != SHARD_COUNT or marker["missing_projects"]:
        raise RuntimeError(f"Invalid/incomplete shard marker: {marker}")
    shard_builds.append(builds[0])
    print(f"Validated shard {index:02d}: {marker['attempted_projects']} projects")
print("All shard build directories:", [str(path) for path in shard_builds])


## Merge shard outputs


In [ ]:
merge_script = PIPELINE_DIR / "merge_shards.py"
run([
    sys.executable, "-u", merge_script,
    "--shard-build-dirs", *shard_builds,
    "--work-dir", MERGED_BUILD,
])


## Finalize contrastive train/validation/test


In [ ]:
prepare = PIPELINE_DIR / "prepare_dataset.py"
run([
    sys.executable, "-u", prepare,
    "--stage", "finalize",
    "--typepro-root", REPO_DIR,
    "--work-dir", MERGED_BUILD,
    "--output-dir", FINAL_DIR,
    "--split-profile", "paper_project",
    "--test-projects", 100,
    "--validation-project-ratio", 0.10,
    "--max-negatives", 7,
    "--seed", SEED,
    "--preview-samples", 2,
    "--preview-max-chars", 1600,
    "--log-every", 10000,
])
run([sys.executable, PIPELINE_DIR / "verify_dataset.py", "--data-dir", FINAL_DIR])


## Display exact counts and examples


In [ ]:
manifest = json.loads((FINAL_DIR / "manifest.json").read_text(encoding="utf-8"))
stats = json.loads((FINAL_DIR / "preprocess_stats.json").read_text(encoding="utf-8"))
print(json.dumps({
    "output": manifest["output"],
    "prepared_counts": manifest["split"]["prepared_counts"],
    "prepared_projects": manifest["split"]["prepared_projects"],
    "preprocess_stats": stats,
}, indent=2, ensure_ascii=False))
for split in ("train", "validation", "test"):
    print(f"\n===== {split.upper()} SAMPLES =====")
    with (FINAL_DIR / f"{split}.jsonl").open(encoding="utf-8") as handle:
        for _, line in zip(range(2), handle):
            print(json.dumps(json.loads(line), indent=2, ensure_ascii=False)[:3000])


## Publish final private Kaggle Dataset


In [ ]:
final_id = f"{os.environ['KAGGLE_USERNAME']}/{FINAL_DATASET_SLUG}"
run([
    sys.executable, PIPELINE_DIR / "publish_kaggle.py",
    "--data-dir", FINAL_DIR,
    "--dataset-id", final_id,
    "--title", "TypePro Python Parameter Third-Party Contrastive Dataset",
    "--message", f"Merge {SHARD_COUNT} verified TypePro shards",
])
completion = {
    "dataset_id": final_id,
    "shard_count": SHARD_COUNT,
    "output": manifest["output"],
}
(FINAL_DIR / "MERGE_COMPLETE.json").write_text(
    json.dumps(completion, indent=2), encoding="utf-8"
)
print(json.dumps(completion, indent=2))
